# Public-source note

Canonical starting-cell RNA clustering notebook. The final analysis uses **K=4**. It constructs cell-type-associated expression clusters and ±1 kb TSS BED intervals. The historical folder name containing `k5` was a naming error and is not a K=5 analysis. Outputs were stripped and paths sanitized.

In [ ]:

library(DESeq2)
library(readr)
library(dplyr)
library(pheatmap)
library(ggplot2)
library(stringr)
library(factoextra)
library(tidyr)
library(ggpubr)
# library(btools)


In [ ]:
readspergene_dir="/path/to/cebp_seq/bulk_rna/5_readspergene_out"
output_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types"
plot_dir="/path/to/CEBPB_seq/plots/bulk_rna/Cluster_all_starting_cell_types"

In [ ]:
samples <- list.files(path = readspergene_dir, 
                full.names = FALSE, 
                pattern="ReadsPerGene.out.tab")


# Ensure the object is treated as character (not factor)
samples <- as.character(samples)

In [ ]:
samples

In [ ]:
groups_to_keep <- c(
    "MEF_Rep",
    "NSC_rep",
    'ESC_Rep',
    "Bcell_Rep")


    

In [ ]:
# Keep only samples containing any of these substrings
samples_filt <- samples[grepl(paste(groups_to_keep, collapse = "|"), samples)]

In [ ]:
samples_filt

In [ ]:
length(samples_filt)

In [ ]:
length(groups_to_keep)

In [ ]:

# Clean up sample names
clean_names <- gsub('"', '', samples_filt)         # remove any stray double-quote characters
clean_names <- trimws(clean_names)            # remove leading/trailing whitespace
clean_names <- sub("^(.*[Rr]ep[0-9]+).*",     # keep everything up to "repX" or "RepX"
                   "\\1",
                   clean_names)
clean_names <- sub("rep([0-9]+)",             # normalize to "RepX" (capital R)
                   "Rep\\1",
                   clean_names,
                   ignore.case = TRUE)

# Assign cleaned sample names to the files vector
# -> now names(samples) will be "Sample_Rep1", "Sample_Rep2", etc.
names(samples_filt) <- clean_names

In [ ]:
samples_filt

In [ ]:
# Read gene lengths table (tab-delimited) into R
gene.lengths <- read_delim("/path/to/refdata/igenomes/index/GRCm39/GRCm39_gencodevM33_geneLengths.txt", "\t",
                           trim_ws = TRUE, escape_double = TRUE)

In [ ]:
for (fi in seq_along(samples_filt)) {
  fname <- samples_filt[fi]

  dir_to_use <- if (any(startsWith(fname, c("Beige", "WAT", "BAT")))) {
    Adipo_GSE198107_readspergene_dir
  } else {
    readspergene_dir
  }

  data <- read_delim(
    file.path(dir_to_use, fname),
    "\t",
    trim_ws = TRUE,
    escape_double = TRUE,
    col_names = c("gene_id", "unstranded", "firststrand", "secondstrand"),
    skip = 4
  )

  cor_first  <- cor(data$unstranded, data$firststrand)
  cor_second <- cor(data$unstranded, data$secondstrand)

  # decide which strand to use
  strand_to_use <- if (cor_first > 0.9 && cor_second > 0.9) {
    "unstranded"
  } else if (cor_first > cor_second) {
    "firststrand"
  } else {
    "secondstrand"
  }

  message(sprintf("[%s] Using %s (cor_first=%.3f, cor_second=%.3f)",
                  fname, strand_to_use, cor_first, cor_second))

  # you can now use: data[[strand_to_use]] for downstream analysis
}


In [ ]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(purrr)
})

choose_dir <- function(fname) {
  if (any(startsWith(fname, c("Beige", "WAT", "BAT")))) {
    Adipo_GSE198107_readspergene_dir
  } else if (startsWith(fname, "chronis_cell")) {
    chronis_cell_readspergene_dir
  } else {
    readspergene_dir
  }
}

pick_strand <- function(df) {
  cor_first  <- cor(df$unstranded, df$firststrand)
  cor_second <- cor(df$unstranded, df$secondstrand)
  if (cor_first > 0.9 && cor_second > 0.9) "unstranded" else if (cor_first > cor_second) "firststrand" else "secondstrand"
}

id_from_fname <- function(fname) sub("\\.out\\.tab$", "", fname)

parts <- vector("list", length(samples_filt))

for (fi in seq_along(samples_filt)) {
  fname <- samples_filt[fi]
  dir_i <- choose_dir(fname)

  df <- read_delim(
    file.path(dir_i, fname),
    "\t",
    trim_ws = TRUE,
    escape_double = TRUE,
    col_names = c("gene_id", "unstranded", "firststrand", "secondstrand"),
    skip = 4
  )

  strand <- pick_strand(df)

  data <- df %>%
    select(gene_id, all_of(strand))

  # keep this exact renaming style, with a safe fallback
  colnames(data)[2] <- if (!is.null(names(samples_filt)) && nzchar(names(samples_filt)[fi])) {
    names(samples_filt)[fi]
  } else {
    id_from_fname(fname)
  }

  parts[[fi]] <- data
}

# combine all by gene_id
sample_data <- reduce(parts, full_join, by = "gene_id")


In [ ]:
head(sample_data)

In [ ]:
# Attach gene_name to counts by gene_id
sample_data <- left_join(sample_data,
                         dplyr::select(gene.lengths, gene_id, gene_name),
                         by = "gene_id")

# Identify count columns programmatically
count_cols <- setdiff(names(sample_data), c("gene_id", "gene_name"))

# Compute total counts per gene across all samples
sample_data <- sample_data %>%
  mutate(total = rowSums(across(all_of(count_cols), as.numeric), na.rm = TRUE))

# For gene symbols that map to multiple gene_ids, keep the row with the highest total
# top_n is deprecated, use slice_max
sample_data <- sample_data %>%
  group_by(gene_name) %>%
  slice_max(order_by = total, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  select(-total, -gene_id)

In [ ]:
head(sample_data)

In [ ]:
nrow(sample_data)

In [ ]:
# number of rows with any NA
sum(!complete.cases(sample_data))


In [ ]:
cat("Rows with NA:", sum(!complete.cases(sample_data)), "\n")
cat("Cols with NA:", sum(colSums(is.na(sample_data)) > 0), "\n")


In [ ]:

# Identify count columns programmatically
count_cols <- setdiff(names(sample_data), c("gene_id", "gene_name"))

# Build numeric matrix with gene_name as rownames
data_mat <- as.matrix(select(sample_data, all_of(count_cols)))
rownames(data_mat) <- sample_data$gene_name


# Define your metadata
meta.data <- data.frame(sample_rep = colnames(data_mat))

# Remove replicate part (_Rep1, _Rep2, etc.) to get the biological sample ID
meta.data$sample <- str_replace(meta.data$sample_rep, "_Rep[0-9]+$", "")

# Optional: extract replicate number as a separate column
meta.data$replicate <- str_extract(meta.data$sample_rep, "Rep[0-9]+")

# Make sure columns and rows of metadata are in the exact same order
all(meta.data$sample_rep == colnames(data_mat))


#    Filter out very lowly expressed genes
#    Keep genes with total counts > 2 across all samples
data_mat <- data_mat[rowSums(data_mat) > 2, , drop = FALSE]

#    Make sure metadata rows match columns of the count matrix
stopifnot(identical(meta.data$sample_rep, colnames(data_mat)))

#    Use a clean design column
#    Convert 'sample' to a factor so DESeq2 treats it as categorical
meta.data$sample <- factor(meta.data$sample)

In [ ]:
meta.data

In [ ]:
#   Build DESeq2 dataset and run the pipeline
dds <- DESeqDataSetFromMatrix(countData = round(data_mat),
                              colData   = meta.data,
                              design    = ~ sample)

dds <- DESeq(dds)



In [ ]:
## DESeq2_normalized_counts
norm_counts <- counts(dds, normalized=T)
write.csv(norm_counts, file.path( output_dir, paste0("Starting_cell_types_DESeq2_normalized_counts.csv")), quote=F)


          
### VST normalized counts
vst_counts <- vst(dds, blind=FALSE)
vst_counts_df <- assay(vst_counts)
write.csv(vst_counts_df, file.path( output_dir,paste0("Starting_cell_types_DESeq2_VST_counts.csv")), quote=F)


In [ ]:
library(dplyr)
library(readr)
library(stringr)
library(tidyr)

## 1) Build a clean gene_name -> gene_length_bp lookup
# try common length column names in your file
len_col_candidates <- c("gene_length", "length", "length_bp", "bp", "tx_length", "geneLen")
len_col <- intersect(len_col_candidates, names(gene.lengths))
stopifnot(length(len_col) >= 1)
len_col <- len_col[1]  # take the first match

length_tbl <- gene.lengths |>
  select(gene_name, gene_length_bp = all_of(len_col)) |>
  filter(!is.na(gene_name), !is.na(gene_length_bp)) |>
  group_by(gene_name) |>
  summarize(gene_length_bp = median(as.numeric(gene_length_bp), na.rm = TRUE), .groups = "drop")

# lengths aligned to your count matrix rows
len_vec_bp <- length_tbl$gene_length_bp[match(rownames(data_mat), length_tbl$gene_name)]
names(len_vec_bp) <- rownames(data_mat)

# drop genes with missing or zero length
keep_len <- !is.na(len_vec_bp) & len_vec_bp > 0
if (!all(keep_len)) {
  message("Dropping ", sum(!keep_len), " genes with missing or zero length before TPM")
}
data_mat_len <- data_mat[keep_len, , drop = FALSE]
len_vec_kb   <- (len_vec_bp[keep_len]) / 1000

## 2) RPK and per sample scaling
# RPK = counts divided by gene length in kilobases
rpk <- sweep(data_mat_len, 1, len_vec_kb, "/")

# per sample scaling factor is sum of RPKs divided by one million
scale_factors <- colSums(rpk, na.rm = TRUE) / 1e6

## 3) TPM
tpm <- sweep(rpk, 2, scale_factors, "/")

## 4) Optional sanity checks
stopifnot(all(abs(colSums(tpm) - 1e6) < 1e-6 | is.na(colSums(tpm))))  # each column sums to one million

## 5) Save
# write.csv(as.data.frame(tpm), file.path(output_dir, paste0(cell_type, "_TPM.csv")), quote = FALSE)


## Differential expression analysis

In [ ]:
suppressPackageStartupMessages({
  library(DESeq2)
  library(dplyr)
  library(tibble)
  library(purrr)
  library(stringr)
  library(pheatmap)
  library(RColorBrewer)
})

## PARAMETERS
alpha        <- 0.05
lfc_cut      <- 1.0           # absolute log2 fold change cutoff
min_base_mean <- 10           # minimum expression threshold on baseMean, adjust if needed
K_clusters   <- NULL          # if NULL will use number of samples as K

## SAMPLES AND PAIRS
samples <- levels(colData(dds)$sample)
stopifnot(length(samples) >= 2)
pairs <- combn(samples, 2, simplify = FALSE)

## RUN ALL PAIRWISE CONTRASTS WITH OPTIONAL LFC SHRINK
res_list <- list()

up_in <- setNames(vector("list", length(samples)), samples)
for (s in samples) up_in[[s]] <- setNames(vector("list", length(setdiff(samples, s))), setdiff(samples, s))

for (pr in pairs) {
  A <- pr[1]; B <- pr[2]

  res_ab <- results(dds, contrast = c("sample", A, B), alpha = alpha)


  res_ab <- lfcShrink(
  dds,
  contrast = c("sample", A, B),
  res = res_ab,
  type = "ashr"
)

    
  # suppressWarnings({
  #   res_ab <- tryCatch(
  #     lfcShrink(dds, contrast = c("sample", A, B), res = res_ab, type = "apeglm"),
  #     error = function(e1) tryCatch(
  #       lfcShrink(dds, contrast = c("sample", A, B), res = res_ab, type = "ashr"),
  #       error = function(e2) res_ab
  #     )

  df <- as.data.frame(res_ab) %>%
    rownames_to_column("gene") %>%
    mutate(
      contrast = paste(A, "vs", B),
      pass = !is.na(padj) & padj <= alpha & baseMean >= min_base_mean & abs(log2FoldChange) >= lfc_cut,
      direction = case_when(
        pass & log2FoldChange >=  lfc_cut ~ paste0("up_in_", A),
        pass & log2FoldChange <= -lfc_cut ~ paste0("up_in_", B),
        TRUE ~ "none"
      )
    )

  res_list[[paste(A, "vs", B)]] <- df

  up_in[[A]][[B]] <- df$gene[df$pass & df$log2FoldChange >=  lfc_cut]
  up_in[[B]][[A]] <- df$gene[df$pass & df$log2FoldChange <= -lfc_cut]
}

all_res <- bind_rows(res_list)

## STEP 1: all pairwise results are in res_list and all_res
## STEP 2: filters already applied via alpha, lfc_cut, min_base_mean in the loop above



In [ ]:
head(all_res)

In [ ]:
## STEP 3: union of all significant genes across every comparison
union_genes <- unique(all_res$gene[all_res$pass])
message("Union size: ", length(union_genes))

## FIND SAMPLE SPECIFIC GENES
## specific to sample S means S is up vs every other sample
sample_specific <- lapply(samples, function(S) {
  others <- setdiff(samples, S)
  sets <- up_in[[S]][others]
  if (length(sets) == 0) character(0) else Reduce(intersect, sets)
})
names(sample_specific) <- samples

In [ ]:

## FIND GENES SHARED BETWEEN EXACTLY TWO SAMPLES
## shared for S1 and S2 means S1 is up vs all others except S2, and S2 is up vs all others except S1
## plus S1 and S2 are not significantly different from each other
shared_two <- list()
for (i in 1:(length(samples) - 1)) {
  for (j in (i + 1):length(samples)) {
    S1 <- samples[i]; S2 <- samples[j]
    others <- setdiff(samples, c(S1, S2))

    s1_up_all <- if (length(others) == 0) character(0) else Reduce(intersect, up_in[[S1]][others])
    s2_up_all <- if (length(others) == 0) character(0) else Reduce(intersect, up_in[[S2]][others])

    df12 <- res_list[[paste(S1, "vs", S2)]]
    neutral_12 <- df12$gene[
      !is.na(df12$padj) & df12$baseMean >= min_base_mean &
        (df12$padj > alpha | abs(df12$log2FoldChange) < lfc_cut)
    ]

    shared <- intersect(intersect(s1_up_all, s2_up_all), neutral_12)
    shared_two[[paste(S1, "+", S2)]] <- shared
  }
}

## SUMMARIES
specific_counts <- sapply(sample_specific, length)
shared_counts   <- sapply(shared_two, length)
print(specific_counts)


In [ ]:
shared_counts


In [ ]:
## FIND GENES SHARED BETWEEN EXACTLY THREE SAMPLES
shared_three <- list()

get_neutral <- function(A, B, res_list, min_base_mean, alpha, lfc_cut) {
  df <- res_list[[paste(A, "vs", B)]]
  df$gene[!is.na(df$padj) & df$baseMean >= min_base_mean &
            (df$padj > alpha | abs(df$log2FoldChange) < lfc_cut)]
}

n <- length(samples)
for (i in 1:(n - 2)) {
  for (j in (i + 1):(n - 1)) {
    for (k in (j + 1):n) {
      S1 <- samples[i]; S2 <- samples[j]; S3 <- samples[k]
      others <- setdiff(samples, c(S1, S2, S3))

      s1_up_all <- if (length(others) == 0) character(0) else Reduce(intersect, up_in[[S1]][others])
      s2_up_all <- if (length(others) == 0) character(0) else Reduce(intersect, up_in[[S2]][others])
      s3_up_all <- if (length(others) == 0) character(0) else Reduce(intersect, up_in[[S3]][others])

      neutral_12 <- get_neutral(S1, S2, res_list, min_base_mean, alpha, lfc_cut)
      neutral_13 <- get_neutral(S1, S3, res_list, min_base_mean, alpha, lfc_cut)
      neutral_23 <- get_neutral(S2, S3, res_list, min_base_mean, alpha, lfc_cut)

      shared <- Reduce(intersect, list(s1_up_all, s2_up_all, s3_up_all,
                                       neutral_12, neutral_13, neutral_23))
      shared_three[[paste(S1, "+", S2, "+", S3)]] <- shared
    }
  }
}

## SUMMARIES
specific_counts    <- sapply(sample_specific, length)
shared_two_counts  <- sapply(shared_two, length)
shared_three_counts <- sapply(shared_three, length)




In [ ]:
shared_three_counts

In [ ]:
## 0) basic checks
stopifnot(is(dds, "DESeqDataSet"))
stopifnot("sample" %in% colnames(colData(dds)))
stopifnot(length(union_genes) > 0)

## 1) VST and replicate averaging
vsd <- vst(dds, blind = FALSE)
mat <- assay(vsd)

sample_ids  <- as.character(colData(vsd)$sample)
samples_all <- unique(sample_ids)
samples     <- samples_all

mat_by_sample <- sapply(samples, function(s) {
  rowMeans(mat[, sample_ids == s, drop = FALSE])
})

## 2) subset to union genes and z score, drop constant rows
keep_genes <- intersect(rownames(mat_by_sample), union_genes)
mat_union  <- mat_by_sample[keep_genes, , drop = FALSE]

# drop constant rows before scale
row_sd <- apply(mat_union, 1, sd)
mat_union <- mat_union[row_sd > 0, , drop = FALSE]

mat_z <- t(scale(t(mat_union)))  # no NA now, since constant rows removed

## 3) desired column order
desired_order <- c(
    "MEF",  
    "NSC", 
    "ESC",
    "Bcell"  
    )
    


present <- desired_order[desired_order %in% colnames(mat_z)]
if (length(present) == 0) stop("None of desired_order are present in the matrix")
mat_z <- mat_z[, present, drop = FALSE]
sample_names <- colnames(mat_z)

## 4) column annotation and colors
ann_col <- data.frame(sample = factor(sample_names, levels = sample_names))
rownames(ann_col) <- sample_names

pal_master <- c(
    "MEF"                          = "#7F7F7F",
        "NSC"                          = "#7F7F7F",
            "ESC"                          = "#7F7F7F",
            "Bcell"                          = "#7F7F7F"

                
)
sample_cols_vec <- pal_master[sample_names]
if (any(is.na(sample_cols_vec))) stop("Missing colors for some sample names")

## 5) k means and cluster ordering
# K_clusters <- 
# if (nrow(mat_z) < K_clusters) {
#   K_clusters <- max(2L, nrow(mat_z))
# }

for (K_clusters in 2:20) {
    
    set.seed(7)
    km <- kmeans(mat_z, centers = K_clusters, nstart = 50, iter.max = 100)
    
    centroids <- rowsum(mat_z, group = km$cluster) / as.vector(table(km$cluster))
    clus_ids  <- as.integer(rownames(centroids))
    
    pref_idx  <- max.col(centroids, ties.method = "first")
    pref_name <- colnames(centroids)[pref_idx]
    strength  <- centroids[cbind(seq_len(nrow(centroids)), pref_idx)]
    
    pref_tbl <- data.frame(
      cluster_id = clus_ids,
      pref_sample = pref_name,
      pref_rank = match(pref_name, sample_names),
      strength = strength,
      stringsAsFactors = FALSE
    )
    
    ord_clusters <- pref_tbl$cluster_id[order(pref_tbl$pref_rank, -pref_tbl$strength)]
    
    ## 6) reorder rows and build a consistent row annotation with sequential labels
    row_order <- unlist(lapply(ord_clusters, function(cl) which(km$cluster == cl)))
    mat_ord   <- mat_z[row_order, , drop = FALSE]
    
    # map original cluster ids to sequential labels 1..n in the chosen order
    n_clu <- length(ord_clusters)
    seq_labels_map <- setNames(seq_len(n_clu), as.character(ord_clusters))
    clusters_in_order_seq <- seq_labels_map[as.character(km$cluster[row_order])]
    
    ann_row <- data.frame(cluster = factor(clusters_in_order_seq, levels = seq_len(n_clu)))
    rownames(ann_row) <- rownames(mat_ord)

    ## -------------------------
    ## PATCH: add gene set annotation
    ## -------------------------
    in_set <- rownames(mat_ord) %in% sample_specific$MEF
    ann_row$MEF_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))
    
    in_set <- rownames(mat_ord) %in% sample_specific$NSC
    ann_row$NSC_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))

    in_set <- rownames(mat_ord) %in% sample_specific$ESC
    ann_row$ESC_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))

    in_set <- rownames(mat_ord) %in% sample_specific$Bcell
    ann_row$Bcell_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))



   
    
    # colors must be named by the factor levels, which are 1..n
    # base_cols <- RColorBrewer::brewer.pal(min(8, max(3, n_clu)), "Set2")
    # cluster_cols_vec <- colorRampPalette(base_cols)(n_clu)
    # cluster_cols_vec <- brewer.pal(n_clu, "Paired")
    # names(cluster_cols_vec) <- as.character(seq_len(n_clu))
    
    base_cols <- RColorBrewer::brewer.pal(min(12, max(3, n_clu)), "Paired")
    cluster_cols_vec <- colorRampPalette(base_cols)(n_clu)
    names(cluster_cols_vec) <- as.character(seq_len(n_clu))
    
    

    
    ann_colors <- list(
        cluster = cluster_cols_vec,
        sample  = setNames(sample_cols_vec, names(sample_cols_vec)),
        MEF_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
        NSC_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
        ESC_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
        Bcell_specific_genes = c(Yes = "#D62728", No = "#E5E5E5")  # red = in set, grey = not

    )
    
    ## 7) heatmap colors and plot
    palette_length <- 101
    my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(palette_length)
    # my_colors <- colorRampPalette(c("blue", "white", "red"))(101)
    # my_colors <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(101)
    # library(viridis)
    # my_colors <- viridis(101, option = "plasma")
    # library(RColorBrewer)
    # my_colors <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(101)
    # my_colors <- colorRampPalette(rev(brewer.pal(11, "PuOr")))(101)
    
    
    
    
    
    options(repr.plot.width = 10, repr.plot.height = 15)
    p <- pheatmap(
      mat_ord,
      cluster_rows = FALSE,
      cluster_cols = FALSE,
      annotation_row = ann_row,
      annotation_col = ann_col,
      annotation_colors = ann_colors,
      color = my_colors,
      # breaks = seq(-2.5, 2.5, length.out = palette_length),
        breaks = seq(-2, 2, length.out = palette_length),
      show_rownames = FALSE,
      fontsize_col = 15,
      fontsize = 15,
      fontsize_legend = 14,
      main = paste0(
        "Union of DE genes (#genes=", nrow(mat_ord), ")\n",
        "k means clusters (k=", K_clusters, ")\n",
          "Color scale: row-scaled VST expression (z-score)"
      )
    )
    
    print(p)
   }
                               
    # dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)
    # outfile <- file.path(plot_dir, sprintf("MEF_RNA_seq_k-means_clusters_k%d.pdf", K_clusters))
    # ggsave(outfile, p$gtable, width = 9, height = 15, dpi = 300)
    # outfile <- file.path(plot_dir, sprintf("MEF_RNA_seq_k-means_clusters_k%d.png", K_clusters))
    # ggsave(outfile, p$gtable, width = 9, height = 15, dpi = 300)
    
    # message("Saved: ", outfile, "   exists = ", file.exists(outfile))
                               

In [ ]:
length(ord_clusters)

In [ ]:
K_clusters <- 4
K_clusters

In [ ]:
## 0) basic checks
stopifnot(is(dds, "DESeqDataSet"))
stopifnot("sample" %in% colnames(colData(dds)))
stopifnot(length(union_genes) > 0)

## 1) VST and replicate averaging
vsd <- vst(dds, blind = FALSE)
mat <- assay(vsd)

sample_ids  <- as.character(colData(vsd)$sample)
samples_all <- unique(sample_ids)
samples     <- samples_all

mat_by_sample <- sapply(samples, function(s) {
  rowMeans(mat[, sample_ids == s, drop = FALSE])
})

## 2) subset to union genes and z score, drop constant rows
keep_genes <- intersect(rownames(mat_by_sample), union_genes)
mat_union  <- mat_by_sample[keep_genes, , drop = FALSE]

# drop constant rows before scale
row_sd <- apply(mat_union, 1, sd)
mat_union <- mat_union[row_sd > 0, , drop = FALSE]

mat_z <- t(scale(t(mat_union)))  # no NA now, since constant rows removed

## 3) desired column order
desired_order <- c(
    "MEF",  
    "NSC", 
    "ESC",
    "Bcell"  
    )
    


present <- desired_order[desired_order %in% colnames(mat_z)]
if (length(present) == 0) stop("None of desired_order are present in the matrix")
mat_z <- mat_z[, present, drop = FALSE]
sample_names <- colnames(mat_z)

## 4) column annotation and colors
ann_col <- data.frame(sample = factor(sample_names, levels = sample_names))
rownames(ann_col) <- sample_names

pal_master <- c(
    "MEF"                          = "#7F7F7F",
        "NSC"                          = "#7F7F7F",
            "ESC"                          = "#7F7F7F",
            "Bcell"                          = "#7F7F7F"

                
)
sample_cols_vec <- pal_master[sample_names]
if (any(is.na(sample_cols_vec))) stop("Missing colors for some sample names")


set.seed(7)
km <- kmeans(mat_z, centers = K_clusters, nstart = 50, iter.max = 100)

centroids <- rowsum(mat_z, group = km$cluster) / as.vector(table(km$cluster))
clus_ids  <- as.integer(rownames(centroids))

pref_idx  <- max.col(centroids, ties.method = "first")
pref_name <- colnames(centroids)[pref_idx]
strength  <- centroids[cbind(seq_len(nrow(centroids)), pref_idx)]

pref_tbl <- data.frame(
  cluster_id = clus_ids,
  pref_sample = pref_name,
  pref_rank = match(pref_name, sample_names),
  strength = strength,
  stringsAsFactors = FALSE
)

ord_clusters <- pref_tbl$cluster_id[order(pref_tbl$pref_rank, -pref_tbl$strength)]

## 6) reorder rows and build a consistent row annotation with sequential labels
row_order <- unlist(lapply(ord_clusters, function(cl) which(km$cluster == cl)))
mat_ord   <- mat_z[row_order, , drop = FALSE]

# map original cluster ids to sequential labels 1..n in the chosen order
n_clu <- length(ord_clusters)
seq_labels_map <- setNames(seq_len(n_clu), as.character(ord_clusters))
clusters_in_order_seq <- seq_labels_map[as.character(km$cluster[row_order])]

ann_row <- data.frame(cluster = factor(clusters_in_order_seq, levels = seq_len(n_clu)))
rownames(ann_row) <- rownames(mat_ord)

## -------------------------
## PATCH: add gene set annotation
## -------------------------
in_set <- rownames(mat_ord) %in% sample_specific$MEF
ann_row$MEF_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))

in_set <- rownames(mat_ord) %in% sample_specific$NSC
ann_row$NSC_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))

in_set <- rownames(mat_ord) %in% sample_specific$ESC
ann_row$ESC_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))

in_set <- rownames(mat_ord) %in% sample_specific$Bcell
ann_row$Bcell_specific_genes <- factor(ifelse(in_set, "Yes", "No"), levels = c("Yes","No"))





# colors must be named by the factor levels, which are 1..n
# base_cols <- RColorBrewer::brewer.pal(min(8, max(3, n_clu)), "Set2")
# cluster_cols_vec <- colorRampPalette(base_cols)(n_clu)
# cluster_cols_vec <- brewer.pal(n_clu, "Paired")
# names(cluster_cols_vec) <- as.character(seq_len(n_clu))

base_cols <- RColorBrewer::brewer.pal(min(12, max(3, n_clu)), "Paired")
cluster_cols_vec <- colorRampPalette(base_cols)(n_clu)
names(cluster_cols_vec) <- as.character(seq_len(n_clu))




ann_colors <- list(
    cluster = cluster_cols_vec,
    sample  = setNames(sample_cols_vec, names(sample_cols_vec)),
    MEF_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
    NSC_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
    ESC_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
    Bcell_specific_genes = c(Yes = "#D62728", No = "#E5E5E5")  # red = in set, grey = not

)

## 7) heatmap colors and plot
palette_length <- 101
my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(palette_length)
# my_colors <- colorRampPalette(c("blue", "white", "red"))(101)
# my_colors <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(101)
# library(viridis)
# my_colors <- viridis(101, option = "plasma")
# library(RColorBrewer)
# my_colors <- colorRampPalette(rev(brewer.pal(11, "RdBu")))(101)
# my_colors <- colorRampPalette(rev(brewer.pal(11, "PuOr")))(101)





options(repr.plot.width = 10, repr.plot.height = 15)
p <- pheatmap(
  mat_ord,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = ann_row,
  annotation_col = ann_col,
  annotation_colors = ann_colors,
  color = my_colors,
  # breaks = seq(-2.5, 2.5, length.out = palette_length),
    breaks = seq(-2, 2, length.out = palette_length),
  show_rownames = FALSE,
  fontsize_col = 15,
  fontsize = 15,
  fontsize_legend = 14,
  main = paste0(
    "Union of DE genes (#genes=", nrow(mat_ord), ")\n",
    "k means clusters (k=", K_clusters, ")\n",
      "Color scale: row-scaled VST expression (z-score)"
  )
)

print(p)


In [ ]:
library(dplyr)
library(tidyr)

## anno_ord must match mat_ord rows
anno_ord <- ann_row[rownames(mat_ord), , drop = FALSE]

## original cluster per displayed row
cluster_orig <- droplevels(anno_ord$cluster)   # "C1", "C2", ...

## wide table in the same row order as the heatmap
tbl_wide <- data.frame(
  gene = rownames(mat_ord),
  cluster_original = as.character(cluster_orig),
  mat_ord,
  check.names = FALSE
)

## long table
tbl_long <- tbl_wide |>
  pivot_longer(
    cols = -c(gene, cluster_original),
    names_to = "sample",
    values_to = "zscore"
  )

## genes in C1 in the same display order
genes_C1_in_display_order <- rownames(mat_ord)[anno_ord$cluster == "1"]

head(tbl_wide, 5)
head(tbl_long, 5)
genes_C1_in_display_order[1:10]


In [ ]:
for (cl in sort(unique(tbl_wide$cluster_original))) {
  n_genes <- length(subset(tbl_wide, cluster_original == cl)$gene)
  message("Cluster ", cl, ": ", n_genes, " genes")
}


In [ ]:
# all genes in cluster 1
genes_cl <- subset(tbl_wide, cluster_original == '4')$gene

# print as newline-separated (easy copy to Metascape)
cat(genes_cl, sep = "\n")

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
})

## ---- 1) Input TPM matrix with replicate columns -----------------------------
## Expecting columns like "..._Rep1", "..._Rep2" matching your list.
## Replace `tpm` with your TPM data frame or matrix (rownames = gene IDs).
stopifnot(exists("tpm"))
stopifnot(!is.null(rownames(tpm)))

## ---- 2) Average replicates to sample means and align to heatmap samples -----
# Map replicate columns to sample base names by dropping trailing "_Rep#"
rep_map <- tibble(
  col = colnames(tpm),
  sample = str_replace(colnames(tpm), "_Rep[0-9]+$", "")
)

# Average replicates
tpm_avg <- as.data.frame(tpm) |>
  tibble::rownames_to_column("gene") |>
  pivot_longer(-gene, names_to = "col", values_to = "TPM") |>
  left_join(rep_map, by = "col") |>
  group_by(gene, sample) |>
  summarize(TPM = mean(TPM, na.rm = TRUE), .groups = "drop") |>
  pivot_wider(names_from = sample, values_from = TPM)

# Keep only samples used in the heatmap and in the same order
present <- intersect(sample_names, setdiff(colnames(tpm_avg), "gene"))
stopifnot(length(present) > 0)
tpm_avg_mat <- tpm_avg |>
  select(gene, all_of(present)) |>
  tibble::column_to_rownames("gene") |>
  as.matrix()

## ---- 3) Build gene to cluster mapping in the sequential order you made ------
# ann_row already has sequential cluster labels 1..K for rows of mat_ord
gene_to_cluster <- data.frame(
  gene = rownames(mat_ord),
  cluster_seq = as.integer(as.character(ann_row$cluster)),
  stringsAsFactors = FALSE
)

## ---- 4) Align genes between TPM and clustering, compute log2(TPM + 1) -------
common_genes <- intersect(rownames(tpm_avg_mat), gene_to_cluster$gene)
stopifnot(length(common_genes) > 0)

tpm_log2 <- log2(tpm_avg_mat[common_genes, present, drop = FALSE] + 1)

gene_to_cluster_sub <- gene_to_cluster[match(common_genes, gene_to_cluster$gene), ]
stopifnot(identical(gene_to_cluster_sub$gene, rownames(tpm_log2)))

## ---- 5) Long table for plotting, with counts per cluster for facet titles ----
df_long <- as.data.frame(tpm_log2) |>
  tibble::rownames_to_column("gene") |>
  pivot_longer(-gene, names_to = "sample", values_to = "log2TPM") |>
  left_join(gene_to_cluster_sub, by = "gene") |>
  mutate(
    sample = factor(sample, levels = sample_names),
    cluster_seq = factor(cluster_seq, levels = sort(unique(cluster_seq)))
  )

# Label as "C# (n = N)"
n_per_cluster <- df_long |>
  distinct(gene, cluster_seq) |>
  count(cluster_seq, name = "n_genes")

lab_tbl <- n_per_cluster |>
  mutate(label = paste0("C", cluster_seq, " (n = ", n_genes, ")")) |>
  tibble::deframe()

## ---- 6) Plot: one figure, one facet per cluster -----------------------------
# Sample colors reused from heatmap
stopifnot(all(levels(df_long$sample) %in% names(sample_cols_vec)))
sample_cols <- sample_cols_vec[levels(df_long$sample)]

# Optional: consistent y range across facets
y_min <- floor(min(df_long$log2TPM, na.rm = TRUE))
y_max <- ceiling(max(df_long$log2TPM, na.rm = TRUE))
# y_max <- 20

p_violin <- ggplot(df_long, aes(x = sample, y = log2TPM, fill = sample)) +
  geom_violin(trim = FALSE, color = "black", size = 0.2) +
  geom_boxplot(width = 0.15, outlier.size = 0.4, alpha = 0.6, color = "black") +
  scale_fill_manual(values = sample_cols, name = "Sample") +
  scale_x_discrete(drop = FALSE) +
  coord_cartesian(ylim = c(y_min, y_max)) +
  labs(
    x = "Sample",
    y = "log2(TPM + 1)"
  ) +
    facet_grid(
      rows = vars(cluster_seq),
      scales = "fixed",
      labeller = labeller(cluster_seq = as_labeller(lab_tbl))
    ) +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    panel.grid.major.x = element_blank(),
    panel.grid.major.y = element_blank(),
    panel.grid.minor.y = element_blank(),
    legend.position = "bottom",
    strip.background = element_rect(fill = "grey90"),
    strip.text = element_text(face = "bold")
  )
options(repr.plot.width = 10, repr.plot.height = 15)
print(p_violin)
# Save
# dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)
# outfile_violin <- file.path(plot_dir, sprintf("MEF_RNA_seq_TPM_violin_by_cluster_k%d.pdf", K_clusters))
# ggsave(outfile_violin, p_violin, width = 4, height = 14, dpi = 300)

# message("Saved: ", outfile_violin, "   exists = ", file.exists(outfile_violin))


### Filter gnes based on significant genes

In [ ]:
library(dplyr)

# ann_row: data frame with rownames = gene_name and column 'cluster'
# sample_specific$MEF: character vector of gene names

## Since cluster#1 is MEF specific. I want to keep only those genes that are MEF significant genes across every comparison
ann_row_filt <- ann_row %>%
  tibble::rownames_to_column("gene_name") %>%
  filter(!(cluster == 1 & !(gene_name %in% sample_specific$MEF))) %>%
  tibble::column_to_rownames("gene_name")


ann_row_filt <- ann_row_filt %>%
  tibble::rownames_to_column("gene_name") %>%
  filter(!(cluster == 2 & !(gene_name %in% sample_specific$NSC))) %>%
  tibble::column_to_rownames("gene_name")


ann_row_filt <- ann_row_filt %>%
  tibble::rownames_to_column("gene_name") %>%
  filter(!(cluster == 3 & !(gene_name %in% sample_specific$ESC))) %>%
  tibble::column_to_rownames("gene_name")

ann_row_filt <- ann_row_filt %>%
  tibble::rownames_to_column("gene_name") %>%
  filter(!(cluster == 4 & !(gene_name %in% sample_specific$Bcell))) %>%
  tibble::column_to_rownames("gene_name")

In [ ]:
nrow(ann_row_filt)

In [ ]:
mat_ord_filt <- mat_ord[rownames(mat_ord) %in% rownames(ann_row_filt), , drop = FALSE]


In [ ]:
nrow(mat_ord_filt)

In [ ]:


base_cols <- RColorBrewer::brewer.pal(min(12, max(3, n_clu)), "Paired")
cluster_cols_vec <- colorRampPalette(base_cols)(n_clu)
names(cluster_cols_vec) <- as.character(seq_len(n_clu))


ann_colors <- list(
    cluster = cluster_cols_vec,
    sample  = setNames(sample_cols_vec, names(sample_cols_vec)),
    MEF_specific_genes = c(Yes = "#D62728", No = "#E5E5E5"),  # red = in set, grey = not
    NSC_specific_genes = c(Yes = "#009E73", No = "#E5E5E5"),  # red = in set, grey = not
    ESC_specific_genes = c(Yes = "#009E73", No = "#E5E5E5"),  # red = in set, grey = not
    Bcell_specific = c(Yes = "#009E73", No = "#E5E5E5")  # red = in set, grey = not
)

## 7) heatmap colors and plot
palette_length <- 101
my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(palette_length)



options(repr.plot.width = 10, repr.plot.height = 15)
p <- pheatmap(
  mat_ord_filt,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  annotation_row = ann_row_filt,
  annotation_col = ann_col,
  annotation_colors = ann_colors,
  color = my_colors,
  breaks = seq(-1.5, 1.5, length.out = palette_length),
    # breaks = seq(-2, 2, length.out = palette_length),
  show_rownames = FALSE,
  fontsize_col = 15,
  fontsize = 15,
  fontsize_legend = 14,
  main = paste0(
    "Union of DE genes with filtering (#genes=", nrow(mat_ord_filt), ")\n",
    "k means clusters (k=", K_clusters, ")\n",
      "Color scale: row-scaled VST expression (z-score)"
  )
)

print(p)

                               
                               

In [ ]:
suppressPackageStartupMessages({
  library(pheatmap)
  library(RColorBrewer)
})

## --- 1) Define distinct color palettes ---------------------------

# Column colors: use your given master palette
sample_cols_vec <- pal_master

# Row cluster colors: choose distinct, vivid hues unrelated to sample palette
# cluster_cols_vec <- grDevices::hcl.colors(n_clu, palette = "Dark 3")
# names(cluster_cols_vec) <- as.character(seq_len(n_clu))

okabe_ito_safe <- c(
  "#0072B2", "#009E73", "#D55E00", "#CC79A7",
  "#56B4E9", "#000000"
)
cluster_cols_vec <- rep(okabe_ito_safe, length.out = n_clu)
names(cluster_cols_vec) <- as.character(seq_len(n_clu))


# Combine into annotation color list
ann_colors <- list(
  cluster = cluster_cols_vec,               # bold hues for row barcode
  sample  = sample_cols_vec,                # biologically meaningful palette
  MEF_specific_genes          = c(Yes = "#D62728", No = "#E5E5E5"),
  ESC_specific_genes          = c(Yes = "#009E73", No = "#E5E5E5"),
  MEF_d12_OSKM_CEBPA_specific = c(Yes = "#009E73", No = "#E5E5E5")
)

## --- 2) Heatmap color scale --------------------------------------

palette_length <- 101
lim <- 1.5
my_colors <- colorRampPalette(c("navy", "white", "firebrick3"))(palette_length)
my_breaks <- seq(-lim, lim, length.out = palette_length)

## --- 3) Row gaps (cluster separators) ----------------------------

gaps_row <- NULL
if ("cluster" %in% colnames(ann_row_filt)) {
  ord_tab <- table(ann_row_filt$cluster)
  if (length(ord_tab) > 1) gaps_row <- cumsum(ord_tab)[seq_len(length(ord_tab) - 1)]
}

## --- 4) Publication-ready heatmap -------------------------------



p <- pheatmap(
  mat_ord_filt,
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  color = my_colors,
  breaks = my_breaks,
  na_col = "grey90",
  border_color = NA,

  annotation_row = ann_row_filt,
  annotation_col = ann_col,
  annotation_colors = ann_colors,
  annotation_legend = TRUE,

  show_rownames = FALSE,
  fontsize = 12,
  fontsize_col = 12,
  fontsize_legend = 11,
  angle_col = 45,
  gaps_row = gaps_row,
  main = paste0(
    "Union of DE genes with filtering (n = ", nrow(mat_ord_filt), ")\n",
    "k-means clusters (k = ", K_clusters, ")\n",
    "Color scale: row-scaled VST expression z-score"
  ),
  legend = TRUE
)

print(p)
outfile_pdf <- file.path(plot_dir, sprintf("Starting_cell_types_k-means_clusters_k%d.pdf", K_clusters))

ggsave(
  filename = outfile_pdf,
  plot     = p$gtable,
  width    = 9,
  height   = 15,
  device   = cairo_pdf,          # better color handling
  bg       = "white"             # match notebook background
  # you can also add: colormodel = "srgb" via grDevices::pdf, see below
)


# outfile <- file.path(plot_dir, sprintf("Starting_cell_types_k-means_clusters_k%d.pdf", K_clusters))
# ggsave(outfile, p$gtable, width = 9, height = 15, dpi = 300)
# outfile <- file.path(plot_dir, sprintf('Starting_cell_types_k-means_clusters_k%d.png', K_clusters))
# ggsave(outfile, p$gtable, width = 9, height = 16, dpi = 300)


In [ ]:
library(dplyr)
library(tidyr)

## anno_ord must match mat_ord rows
anno_ord <- ann_row_filt[rownames(mat_ord_filt), , drop = FALSE]

## original cluster per displayed row
cluster_orig <- droplevels(anno_ord$cluster)   # "C1", "C2", ...

## wide table in the same row order as the heatmap
tbl_wide <- data.frame(
  gene = rownames(mat_ord_filt),
  cluster_original = as.character(cluster_orig),
  mat_ord_filt,
  check.names = FALSE
)

## long table
tbl_long <- tbl_wide |>
  pivot_longer(
    cols = -c(gene, cluster_original),
    names_to = "sample",
    values_to = "zscore"
  )

## genes in C1 in the same display order
genes_C1_in_display_order <- rownames(mat_ord_filt)[anno_ord$cluster == "1"]

head(tbl_wide, 5)
head(tbl_long, 5)
genes_C1_in_display_order[1:10]


In [ ]:
for (cl in sort(unique(tbl_wide$cluster_original))) {
  n_genes <- length(subset(tbl_wide, cluster_original == cl)$gene)
  message("Cluster ", cl, ": ", n_genes, " genes")
}


In [ ]:
library(readr)
outfile <- file.path(output_dir, sprintf("Starting_cell_types_k-means_clusters_k%d.CSV", K_clusters))
write_csv(tbl_wide, outfile)

In [ ]:
# all genes in cluster 1
genes_cl <- subset(tbl_wide, cluster_original == '1')$gene

# print as newline-separated (easy copy to Metascape)
cat(genes_cl, sep = "\n")

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(ggplot2)
})

## ---- 1) Input TPM matrix with replicate columns -----------------------------
## Expecting columns like "..._Rep1", "..._Rep2" matching your list.
## Replace `tpm` with your TPM data frame or matrix (rownames = gene IDs).
stopifnot(exists("tpm"))
stopifnot(!is.null(rownames(tpm)))

## ---- 2) Average replicates to sample means and align to heatmap samples -----
# Map replicate columns to sample base names by dropping trailing "_Rep#"
rep_map <- tibble(
  col = colnames(tpm),
  sample = str_replace(colnames(tpm), "_Rep[0-9]+$", "")
)

# Average replicates
tpm_avg <- as.data.frame(tpm) |>
  tibble::rownames_to_column("gene") |>
  pivot_longer(-gene, names_to = "col", values_to = "TPM") |>
  left_join(rep_map, by = "col") |>
  group_by(gene, sample) |>
  summarize(TPM = mean(TPM, na.rm = TRUE), .groups = "drop") |>
  pivot_wider(names_from = sample, values_from = TPM)

# Keep only samples used in the heatmap and in the same order
present <- intersect(sample_names, setdiff(colnames(tpm_avg), "gene"))
stopifnot(length(present) > 0)
tpm_avg_mat <- tpm_avg |>
  select(gene, all_of(present)) |>
  tibble::column_to_rownames("gene") |>
  as.matrix()

## ---- 3) Build gene to cluster mapping in the sequential order you made ------
# ann_row_filt already has sequential cluster labels 1..K for rows of mat_ord_filt
gene_to_cluster <- data.frame(
  gene = rownames(mat_ord_filt),
  cluster_seq = as.integer(as.character(ann_row_filt$cluster)),
  stringsAsFactors = FALSE
)

## ---- 4) Align genes between TPM and clustering, compute log2(TPM + 1) -------
common_genes <- intersect(rownames(tpm_avg_mat), gene_to_cluster$gene)
stopifnot(length(common_genes) > 0)

tpm_log2 <- log2(tpm_avg_mat[common_genes, present, drop = FALSE] + 1)

gene_to_cluster_sub <- gene_to_cluster[match(common_genes, gene_to_cluster$gene), ]
stopifnot(identical(gene_to_cluster_sub$gene, rownames(tpm_log2)))

## ---- 5) Long table for plotting, with counts per cluster for facet titles ----
df_long <- as.data.frame(tpm_log2) |>
  tibble::rownames_to_column("gene") |>
  pivot_longer(-gene, names_to = "sample", values_to = "log2TPM") |>
  left_join(gene_to_cluster_sub, by = "gene") |>
  mutate(
    sample = factor(sample, levels = sample_names),
    cluster_seq = factor(cluster_seq, levels = sort(unique(cluster_seq)))
  )

# Label as "C# (n = N)"
n_per_cluster <- df_long |>
  distinct(gene, cluster_seq) |>
  count(cluster_seq, name = "n_genes")

lab_tbl <- n_per_cluster |>
  mutate(label = paste0("C", cluster_seq, " (n = ", n_genes, ")")) |>
  tibble::deframe()

## ---- 6) Plot: one figure, one facet per cluster -----------------------------
# Sample colors reused from heatmap
stopifnot(all(levels(df_long$sample) %in% names(sample_cols_vec)))
sample_cols <- sample_cols_vec[levels(df_long$sample)]

# Optional: consistent y range across facets
y_min <- floor(min(df_long$log2TPM, na.rm = TRUE))
y_max <- ceiling(max(df_long$log2TPM, na.rm = TRUE))
# y_max <- 20

p_violin <- ggplot(df_long, aes(x = sample, y = log2TPM, fill = sample)) +
  geom_violin(trim = FALSE, color = "black", size = 0.2) +
  geom_boxplot(width = 0.15, outlier.size = 0.4, alpha = 0.6, color = "black") +
  scale_fill_manual(values = sample_cols, name = "Sample") +
  scale_x_discrete(drop = FALSE) +
  coord_cartesian(ylim = c(y_min, y_max)) +
  labs(
    x = "Sample",
    y = "log2(TPM + 1)"
  ) +
    facet_grid(
      rows = vars(cluster_seq),
      scales = "fixed",
      labeller = labeller(cluster_seq = as_labeller(lab_tbl))
    ) +
  theme_bw(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1),
    panel.grid.major.x = element_blank(),
    panel.grid.major.y = element_blank(),
    panel.grid.minor.y = element_blank(),
    legend.position = "bottom",
    strip.background = element_rect(fill = "grey90"),
    strip.text = element_text(face = "bold")
  )
options(repr.plot.width = 10, repr.plot.height = 15)
print(p_violin)
# Save
# dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)
# outfile_violin <- file.path(plot_dir, sprintf("MEF_RNA_seq_TPM_violin_by_cluster_k%d.pdf", K_clusters))
# ggsave(outfile_violin, p_violin, width = 4, height = 14, dpi = 300)

# message("Saved: ", outfile_violin, "   exists = ", file.exists(outfile_violin))


In [ ]:
## ---- 7) Plot separate file for each cluster -----------------------------

y_max<- 17
unique_clusters <- levels(df_long$cluster_seq)

for (cl in unique_clusters) {
  df_cl <- df_long %>% filter(cluster_seq == cl)

  # title with cluster size
  n_genes_cl <- n_per_cluster %>% filter(cluster_seq == cl) %>% pull(n_genes)
  cl_title <- paste0("Cluster ", cl, " (n = ", n_genes_cl, " genes)")

  p_cl <- ggplot(df_cl, aes(x = sample, y = log2TPM, fill = sample)) +
    geom_violin(trim = FALSE, color = "black", size = 0.2) +
    geom_boxplot(width = 0.15, outlier.size = 0.4, fill = NA, color = "black") +
    scale_fill_manual(values = sample_cols, guide = "none") +   # remove legend
    scale_x_discrete(drop = FALSE) +
    coord_cartesian(ylim = c(y_min, y_max)) +
    labs(
      x = NULL,
      y = "log2(TPM + 1)",
      title = cl_title
    ) +
    theme_bw(base_size = 12) +
    theme(
      axis.text.x  = element_blank(),   # remove tick labels
      axis.ticks.x = element_blank(),   # remove ticks themselves
      panel.grid.major.x = element_blank(),
      panel.grid.major.y = element_blank(),
      panel.grid.minor.y = element_blank(),
      legend.position = "none"
    )

  print(p_cl)

  outfile_cl <- file.path(
    plot_dir,
    sprintf("Starting_cell_types_k_RNA_seq_TPM_violin_cluster%s.pdf", cl)
  )
  ggsave(outfile_cl, p_cl, width = 2, height = 2, dpi = 300)
  message("Saved: ", outfile_cl)
}


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(purrr)
  library(rstatix)
})

## assumes df_long and sample_names already exist
## df_long columns: gene, sample (factor), log2TPM, cluster_seq (factor)

# reference set in your plotting order
samples_ref <- levels(df_long$sample)

# median per cluster per sample for direction
med_tbl <- df_long %>%
  group_by(cluster_seq, sample) %>%
  summarise(median_log2TPM = median(log2TPM, na.rm = TRUE), .groups = "drop")

# run Wilcoxon in each cluster with every sample once as ref
pvals_all <- map_dfr(samples_ref, function(ref_sample) {
  df_long %>%
    group_by(cluster_seq) %>%
    rstatix::pairwise_wilcox_test(
      log2TPM ~ sample,
      ref.group = ref_sample,
      p.adjust.method = "BH"
    ) %>%
    ungroup() %>%
    mutate(ref_sample = ref_sample)
})

# add stars and formatted numbers
pvals_all <- pvals_all %>%
  mutate(
    p_stars = case_when(
      p.adj <= 1e-4 ~ "****",
      p.adj <= 1e-3 ~ "***",
      p.adj <= 0.01 ~ "**",
      p.adj <= 0.05 ~ "*",
      TRUE ~ "ns"
    ),
    p_num = rstatix::p_format(p.adj, accuracy = 0.001)
  )

# add direction via medians (group1 is ref when ref.group is used)
pvals_all <- pvals_all %>%
  left_join(med_tbl, by = c("cluster_seq", "group1" = "sample")) %>%
  rename(median_ref = median_log2TPM) %>%
  left_join(med_tbl, by = c("cluster_seq", "group2" = "sample")) %>%
  rename(median_cmp = median_log2TPM) %>%
  mutate(
    delta_median = median_cmp - median_ref,
    direction = case_when(
      is.na(delta_median) ~ NA_character_,
      delta_median > 0 ~ "up_vs_ref",
      delta_median < 0 ~ "down_vs_ref",
      TRUE ~ "no_change"
    )
  )

# human friendly comparison label
pvals_all <- pvals_all %>%
  mutate(comparison = paste(group1, "vs", group2))

# final tidy columns and ordering
pvals_all <- pvals_all %>%
  select(
    cluster_seq, ref_sample, comparison, group1, group2,
    p, p.adj, p_num, p_stars, statistic, n1, n2,
    median_ref, median_cmp, delta_median, direction
  ) %>%
  arrange(cluster_seq, ref_sample, match(group2, samples_ref))

# preview
head(pvals_all)


In [ ]:
outfile_csv <- file.path(output_dir, sprintf("pvals_all_results_Starting_cell_types_RNA_seq_cluster_k%d.csv", K_clusters))
write.csv(pvals_all, outfile_csv, row.names = FALSE)
message("Saved: ", outfile_csv)


In [ ]:
pvals_all %>%
  filter(
    cluster_seq == 1,
    (ref_sample == "MEF")  )

In [ ]:
nrow(tbl_wide)

In [ ]:
# Turn rownames of k_mean_df into a real column
tpm_df <- as.data.frame(tpm)
tpm_df$gene <- rownames(tpm_df)
rownames(tpm_df) <- NULL

# Now both data frames have a 'gene' column
k_mean_tpm_df <- merge(tbl_wide[c("gene","cluster_original")], tpm_df, by = "gene")
k_mean_tpm_df <- k_mean_tpm_df[order(k_mean_tpm_df$cluster_original), ]
colnames(k_mean_tpm_df)[colnames(k_mean_tpm_df) == "cluster_original"] <- "Cluster"
colnames(k_mean_tpm_df)[colnames(k_mean_tpm_df) == "gene"] <- "Gene_Symbol"


In [ ]:
head(k_mean_tpm_df)

In [ ]:
colnames(k_mean_tpm_df)

In [ ]:
library(readr)
outfile <- file.path(output_dir, sprintf("Starting_cell_types_k-means_clusters_k%d_TPM.csv", K_clusters))
write_csv(k_mean_tpm_df, outfile)

## mm10_lifted_tss_+-20kb_beds

In [ ]:
library(readr)

K_clusters <- 4

infile <- file.path(
  output_dir,
  sprintf("Starting_cell_types_k-means_clusters_k%d.CSV", K_clusters)
)

Starting_cell_types_cluster_df <- read_csv(infile)


In [ ]:
library(data.table)
mm10_lifted_tss_bed <- "/path/to/CEBPB_seq/plots/bulk_rna/D2_samples_V1/vM33_gene_TSS.mm39ToMm10_lifted.bed"
tss_mm10 <- fread(
  mm10_lifted_tss_bed,
  sep = "\t",
  header = FALSE,
  col.names = c("mm10_lifted_tss_chrom", "mm10_lifted_tss_start", "mm10_lifted_tss_end", "gene_id", "mm10_lifted_tss_score", "mm10_lifted_tss_strand")
)

In [ ]:
head(tss_mm10)

In [ ]:
merged_tbl_wide <- left_join(
  Starting_cell_types_cluster_df,   # left dataframe
  tss_mm10,        # right dataframe
  by = c("gene" = "gene_id")  # map left_on -> right_on
)

head(merged_tbl_wide)

In [ ]:
tss_20kb_bed_dir=file.path(output_dir, sprintf("Starting_cell_types_k-means_clusters_k%d_tss_20kb_beds", K_clusters))
dir.create(tss_20kb_bed_dir, recursive = TRUE)

In [ ]:
for (cl in sort(unique(merged_tbl_wide$cluster_original))) {
    cluster_df=subset(merged_tbl_wide, cluster_original == cl)[, c("cluster_original", "mm10_lifted_tss_chrom", "mm10_lifted_tss_start", "mm10_lifted_tss_end", "gene")]
    cluster_df$'mm10_lifted_tss_start+20kb' <- cluster_df$mm10_lifted_tss_start +20000L
    cluster_df$'mm10_lifted_tss_start-20kb' <- cluster_df$mm10_lifted_tss_start -20000L

    # Remove rows where start+20kb is less than zero
    cluster_df <- subset(cluster_df, `mm10_lifted_tss_start-20kb` >= 0)
    print(min(cluster_df$`mm10_lifted_tss_start-20kb`))

    cluster_sub_df <- cluster_df[, c('mm10_lifted_tss_chrom', 'mm10_lifted_tss_start-20kb' , 'mm10_lifted_tss_start+20kb', "gene")]
    
    fwrite(cluster_sub_df, file.path(tss_20kb_bed_dir, sprintf("RNA_cluster_%s.bed", cl)), sep = "\t", col.names = FALSE, quote = FALSE)
    
  # n_genes <- length(subset(merged_tbl_wide, cluster_original == cl)$gene)
  # message("Cluster ", cl, ": ", n_genes, " genes")
}


In [ ]:
tss_100kb_bed_dir=file.path(output_dir, sprintf("Starting_cell_types_k-means_clusters_k%d_tss_1kb_beds", K_clusters))
dir.create(tss_100kb_bed_dir, recursive = TRUE)

In [ ]:
half_window <- 1000L   # 1 kb on each side

for (cl in sort(unique(merged_tbl_wide$cluster_original))) {

  cluster_df <- subset(
    merged_tbl_wide,
    cluster_original == cl
  )[, c("cluster_original", "mm10_lifted_tss_chrom", "mm10_lifted_tss_start", "mm10_lifted_tss_end", "gene")]

  cluster_df$tss_start_plus  <- cluster_df$mm10_lifted_tss_start + half_window
  cluster_df$tss_start_minus <- cluster_df$mm10_lifted_tss_start - half_window

  cluster_df <- subset(cluster_df, tss_start_minus >= 0)

  cluster_sub_df <- cluster_df[, c("mm10_lifted_tss_chrom", "tss_start_minus", "tss_start_plus", "gene")]

  fwrite(
    cluster_sub_df,
    file.path(tss_100kb_bed_dir, sprintf("RNA_cluster_%s.bed", cl)),
    sep = "\t",
    col.names = FALSE,
    quote = FALSE
  )
}


In [ ]:
tss_100kb_bed_dir